# 12. Python 활용 종합 프로젝트 예제

## Goal

- 파일 수집·검증·해시·보고서 저장을 통합합니다.
- 모든 쓰기를 임시 실습 영역으로 제한합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

실행할 때마다 생성되는 임시 디렉터리만 사용합니다.


## Steps

### 파일 메타데이터 보고서 파이프라인

일반 파일만 수집하고 SHA-256과 크기를 JSON으로 원자적 저장합니다.


In [1]:
from hashlib import sha256
import json
import os
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory


def analyze_file(path: Path) -> dict:
    if not path.is_file() or path.is_symlink():
        raise ValueError("일반 파일만 분석할 수 있습니다")
    data = path.read_bytes()
    return {"name": path.name, "size": len(data), "sha256": sha256(data).hexdigest()}


def write_json_atomic(value: dict, output: Path):
    output.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile("w", encoding="utf-8", dir=output.parent, delete=False) as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        temporary = Path(stream.name)
    os.replace(temporary, output)


def run_demo():
    with TemporaryDirectory() as directory:
        root = Path(directory)
        evidence = root / "evidence"
        evidence.mkdir()
        (evidence / "alpha.txt").write_text("alpha", encoding="utf-8")
        (evidence / "beta.bin").write_bytes(b"\x00\x01")
        records = [analyze_file(path) for path in sorted(evidence.iterdir())]
        report = {"count": len(records), "files": records}
        output = root / "results/report.json"
        write_json_atomic(report, output)
        loaded = json.loads(output.read_text(encoding="utf-8"))
        return loaded


capstone_report = run_demo()
print(json.dumps(capstone_report, ensure_ascii=False, indent=2))


{
  "count": 2,
  "files": [
    {
      "name": "alpha.txt",
      "size": 5,
      "sha256": "8ed3f6ad685b959ead7022518e1af76cd816f8e8ec7ccdda1ed4018e8f2223f8"
    },
    {
      "name": "beta.bin",
      "size": 2,
      "sha256": "b413f47d13ee2fe6c845b2ee141af81de858df4ec549a58b7970bb96645bc8d2"
    }
  ]
}


## Checks

파일 수·크기·해시 형식을 확인합니다.


In [2]:
assert capstone_report["count"] == 2
assert [item["name"] for item in capstone_report["files"]] == ["alpha.txt", "beta.bin"]
assert all(len(item["sha256"]) == 64 for item in capstone_report["files"])
print("종합 프로젝트 검사 통과")


종합 프로젝트 검사 통과


## Next Steps

다음 단계에서는 CLI·로그·테스트를 별도 모듈로 분리하고 실제 허용 범위를 문서화합니다.
